In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import LinearRegression # Added because the PDF requires it for final results
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.metrics.pairwise import rbf_kernel

def inverse_transform_price(scaled_preds, min_log_price, max_log_price):
    """
    Reverses the linear scaling [0.1, 0.9] and the logarithm to return to USD ($).
    You will need 'Developer A' to pass you the min and max values of the original log_price.
    """
    # 1. Reverse the min-max scaling to [0.1, 0.9]
    # Formula: S = 0.1 + 0.8 * (x - min) / (max - min)
    # Solving for x: x = min + (S - 0.1) * (max - min) / 0.8
    log_price_preds = min_log_price + ((scaled_preds - 0.1) * (max_log_price - min_log_price) / 0.8)
    
    # 2. Reverse the logarithm (exponential function)
    usd_preds = np.exp(log_price_preds)
    
    return usd_preds

def calculate_mape(y_true_usd, y_pred_usd):
    """
    Calculates the exact mathematical Mean Absolute Percentage Error (MAPE).
    Both inputs must already be in USD.
    """
    # Avoid division by zero just in case (although prices start at $326)
    epsilon = 1e-10 
    mape = 100 * np.mean(np.abs((y_pred_usd - y_true_usd) / (y_true_usd + epsilon)))
    return mape

In [2]:
class LocalKernelRegression(BaseEstimator, RegressorMixin):
    """
    Custom Scikit-Learn compatible estimator for Local Kernel Regression 
    using Nadaraya-Watson and an RBF Kernel.
    """
    def __init__(self, gamma=1.0):
        self.gamma = gamma # Kernel bandwidth parameter
        
    def fit(self, X, y):
        self.X_train_ = np.array(X)
        self.y_train_ = np.array(y)
        return self
        
    def predict(self, X):
        X = np.array(X)
        
        # Calculate the weight matrix using a Gaussian (RBF) kernel
        weights = rbf_kernel(X, self.X_train_, gamma=self.gamma)
        
        # Nadaraya-Watson prediction: weighted sum of y_train
        # Avoid division by zero by adding a small epsilon to the denominator
        sum_weights = np.sum(weights, axis=1) + 1e-10
        y_pred = np.dot(weights, self.y_train_) / sum_weights
        
        return y_pred

In [3]:
def build_grid_searches():
    """
    Prepares and returns the GridSearchCV objects for each model.
    """
    # 1. k-Nearest Neighbors (k-NN)
    knn_params = {
        'n_neighbors': [3, 5, 7, 10, 15],
        'weights': ['uniform', 'distance'],
        'p': [1, 2] # Manhattan vs Euclidean distance
    }
    knn_grid = GridSearchCV(KNeighborsRegressor(), knn_params, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)

    # 2. Local Kernel Regression (LKR)
    lkr_params = {
        'gamma': [0.01, 0.1, 1.0, 10.0, 100.0]
    }
    lkr_grid = GridSearchCV(LocalKernelRegression(), lkr_params, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)

    # 3. Multilayer Neural Network (MLNN-BP)
    mlnn_params = {
        'hidden_layer_sizes': [(50,), (100,), (50, 50)],
        'activation': ['relu', 'tanh'],
        'alpha': [0.0001, 0.001, 0.01],
        'max_iter': [500] # Increased to ensure convergence
    }
    # Using early_stopping to prevent overfitting during Cross-Validation
    mlnn_grid = GridSearchCV(MLPRegressor(early_stopping=True, random_state=42), 
                             mlnn_params, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)

    # 4. Multilinear Regression (MLR) - Added due to final DataFrame requirements
    mlr_grid = LinearRegression() # Linear regression usually does not require GridSearch

    return {
        'MLR': mlr_grid,
        'k-NN': knn_grid,
        'LKR': lkr_grid,
        'MLNN-BP': mlnn_grid
    }

In [4]:
def evaluate_and_visualize(models_dict, X_test, y_test_scaled, df_test_original, min_log, max_log):
    """
    Performs predictions, inverts the scale, calculates MAPE, creates the DataFrame, and visualizes.
    - y_test_scaled: The scaled real prices [0.1, 0.9].
    - df_test_original: The DataFrame with the original test set features.
    """
    # 1. Invert the real test target to USD to correctly calculate MAPE
    y_test_usd = inverse_transform_price(y_test_scaled, min_log, max_log)
    
    # Prepare the results DataFrame with a copy of the original dataset
    df_results = df_test_original.copy()
    
    mapes = {}
    preds_usd_dict = {}
    
    # 2. Make predictions and calculate metrics
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    axes = axes.flatten()
    
    # Column names exactly as requested in the assignment (section 3.f.ii)
    column_mapping = {
        'MLR': 'pred_price_mlr',
        'k-NN': 'pred_price_knn',
        'MLNN-BP': 'pred_price_bp',
        'LKR': 'pred_price_lkr' # Not explicitly asked in 3.f.ii, but added for completeness
    }
    
    for idx, (name, model) in enumerate(models_dict.items()):
        # Prediction (returns values in [0.1, 0.9] scale)
        pred_scaled = model.predict(X_test)
        
        # Inverse transformation to USD
        pred_usd = inverse_transform_price(pred_scaled, min_log, max_log)
        preds_usd_dict[name] = pred_usd
        
        # Add to the final DataFrame
        col_name = column_mapping.get(name, f'pred_price_{name.lower()}')
        df_results[col_name] = pred_usd
        
        # Calculate MAPE
        current_mape = calculate_mape(y_test_usd, pred_usd)
        mapes[name] = current_mape
        print(f"MAPE for {name}: {current_mape:.4f}%")
        
        # 3. Scatter Plot (Real vs Predicted)
        ax = axes[idx]
        ax.scatter(y_test_usd, pred_usd, alpha=0.3, color='blue', edgecolor='k', s=15)
        
        # Ideal line (y = x)
        max_val = max(np.max(y_test_usd), np.max(pred_usd))
        ax.plot([0, max_val], [0, max_val], 'r--', lw=2, label='Perfect Prediction (y=x)')
        
        ax.set_title(f'{name} - Real vs Predicted (MAPE: {current_mape:.2f}%)')
        ax.set_xlabel('Real Price (USD)')
        ax.set_ylabel('Predicted Price (USD)')
        ax.legend()
        ax.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.show()
    
    # Return the exact DataFrame required by the assignment
    return df_results, mapes